# 🚀 3D Rocket Lander — RL + Domain Randomization (Colab)

PPO agent learns to land a 3D rocket with curriculum-based domain randomization (Tesla-style sim-to-real).

**⚠️ কপি-পেস্ট করবেন না!** GitHub-এর view থেকে কপি করলে নতুন লাইন হারিয়ে যায় (`import osos.chdir...` SyntaxError)।

**সবচেয়ে সহজ:** নিচের এক-ক্লিক লিংকে যান — Colab সরাসরি GitHub থেকে notebook খুলবে:

👉 [**Open in Colab (one click)**](https://colab.research.google.com/github/MD-Mushfiqur123/rocket-lander/blob/vibe/3d-domain-randomized-lander-03248f/colab_rocket_lander.ipynb)

বিকল্প (merge-এর পরে): `https://colab.research.google.com/github/MD-Mushfiqur123/rocket-lander/blob/main/colab_rocket_lander.ipynb`

ম্যানুয়াল পদ্ধতি: ফাইলটি **Download raw file** বাটন দিয়ে ডাউনলোড করুন (কপি নয়), তারপর → [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook**।

---

In [ ]:
REPO = 'https://github.com/MD-Mushfiqur123/rocket-lander'BRANCH = 'vibe/3d-domain-randomized-lander-03248f'  # after merge, use 'main'!git clone -b $BRANCH $REPO

In [ ]:
# PyBullet compiles from source here (no py3.12 wheel) — Colab has gcc, takes ~2-3 min!pip install -q -r /content/rocket-lander/requirements.txt

In [ ]:
import osos.chdir('/content/rocket-lander')import shutil; shutil.rmtree('rocket_lander_3d/__pycache__', ignore_errors=True)TRAIN_STEPS = 1_000_000  # 500k quick run; 1.5M+ for full-randomization robustnessprint('working dir:', os.getcwd())

In [ ]:
# Training: curriculum ramps randomization strength 0→1 in the second half# Checkpoint is saved + evaluated every 100k steps!python train.py --steps {TRAIN_STEPS}

In [ ]:
# (demo code inlined — works standalone)
import base64, io, numpy as np
from PIL import Image
from stable_baselines3 import PPO
from rocket_lander_3d.env import RocketLander3DEnv

def render_frame(cid, rocket_id, width=320, height=320):
    pos, _ = cid.getBasePositionAndOrientation(rocket_id)
    view = cid.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=[pos[0], pos[1], max(pos[2]*0.7, 1.0)],
        distance=8, yaw=45, pitch=-25, roll=0, upAxisIndex=2)
    proj = cid.computeProjectionMatrixFOV(fov=60, aspect=1.0, nearVal=0.1, farVal=50)
    img = cid.getCameraImage(width, height, viewMatrix=view, projectionMatrix=proj)[2]
    return np.reshape(img, (height, width, 4))[:, :, :3]

model = PPO.load('rocket_lander_3d_ppo')
env = RocketLander3DEnv(domain_randomization=True, seed=20260101)
env.set_randomization_strength(1.0)
frames, stats = [], []
for ep in range(3):
    obs, _ = env.reset(); ep_reward, done = 0.0, False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(action)
        ep_reward += r; done = term or trunc
        if env.step_count % 4 == 0:
            frames.append(render_frame(env.client_id, env.rocket_id))
    stats.append((info['outcome'], ep_reward))
env.close()

pil = [Image.fromarray(f.astype(np.uint8)) for f in frames]
pil[0].save('demo.gif', format='GIF', save_all=True,
            append_images=pil[1:], duration=80, loop=0)
html = '<img src="data:image/gif;base64,' + base64.b64encode(open('demo.gif','rb').read()).decode() + '" width="320">'
rows = '\n'.join(f'| {i+1} | {s[0] or "timeout"} | {s[1]:.1f} |' for i, s in enumerate(stats))
from IPython.display import display, Markdown, HTML
display(HTML(html)); display(Markdown(f'| Ep | Outcome | Reward |\n|---|---|---|\n{rows}'))

In [ ]:
# 💾 Save the model to your Google Drive (optional)from google.colab import drivedrive.mount('/content/drive')!cp rocket_lander_3d_ppo.zip /content/drive/MyDrive/from google.colab import filesfiles.download('rocket_lander_3d_ppo.zip')  # or download straight to your PC

## 📌 Notes
- **Training time:** ~20–40 min for 1M steps on Colab's free CPU tier
- Progression you'll see: timeouts → crashes (descending) → **landings**
- The model zip saves every 100k steps — you can stop early and still demo it
- নোটবুক ডাউনলোড করে রাখুন — সব কোড একসাথে!